In [ ]:
from sklearn.cluster import AgglomerativeClustering

def drop_correlated_features( X , threshold = 0.9 ):
    """
    Args:
        - X (pd.DataFrame) : n,p feature matrix
        - threshold (float) : absolute correlation threshold group variables
    
    Returns:
        - pd.DataFrame : X with only the selected variables
        - dict : keys are the selected features , values are the list of features in the corresponding feature cluster 
    """
    
    corr_threshold = 0.9

    metric = 1 - X.corr().abs()

    HC = AgglomerativeClustering( n_clusters=None , metric='precomputed', linkage = 'single' , distance_threshold = (1-corr_threshold) )
    HC.fit(metric)

    variable_clusters = pd.Series( HC.labels_  , index = X.columns)

    cluster_to_features = variable_clusters.index.groupby(variable_clusters)

    ## keys are the selected feature in the cluster, values are the list of features in the cluster
    selected_features_to_features = { v[0]:list(v) for v in cluster_to_features.values() }

    return X.loc[:,selected_features_to_features.keys()] ,  selected_features_to_features 

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectPercentile
import numpy as np

## loading data
df_xpr = pd.read_csv("../data/TGCA_BRCA_expression_matrix.TPM.csv.gz" , index_col = 0)

df_clinical = pd.read_csv("../data/TGCA_BRCA_clinical_filtered.small.csv",index_col=0)
df_clinical = pd.get_dummies( df_clinical , drop_first=True)

## y is the poor_diagnosis
y = df_clinical.poor_prognosis

## ensuring the expression data is properly ordered
X_xpr = df_xpr.loc[ :, df_clinical.index].transpose() 

## selecting top 1% most variable genes
VT = SelectPercentile( score_func = lambda x,_ : np.var(x , axis = 0) ,
                       percentile = 1
                     )

X = pd.DataFrame( VT.fit_transform(X_xpr), columns=VT.get_feature_names_out() , index = X_xpr.index )
X , features_to_features_cluster = drop_correlated_features( X , threshold = 0.9 )

# adding age and sex to the set of features
X = pd.concat( [ df_clinical[['demographic.days_to_birth','demographic.sex_at_birth_male']] , X ] , axis=1 )

X.shape

# Sequential Feature Selection

https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SequentialFeatureSelector.html

This needs:
 * `estimator` : an model, **which does not have to expose a coefficient or importance score**
 * a stopping condition: 
     * number of features to select: `n_features_to_select`  
     * until the score does not improve by a given margin: `tol` 
 * a `direction`: forward or backward

Also good to consider:
 * `scoring`
 * `cv`


In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

ppl = Pipeline([('scale' , StandardScaler()),
                ('model' , KNeighborsClassifier())
               ])


forward_selector = SequentialFeatureSelector(ppl,
                                             tol = 0.01,
                                             direction = 'forward',
                                             scoring = 'accuracy',
                                             cv = 5                                             
                                            )


expect ~30s:

In [ ]:
%time forward_selector.fit(X, y)

In [ ]:
forward_selector.get_support().sum()

In [ ]:
forward_selector.get_feature_names_out()

In [ ]:
from sklearn.model_selection import cross_val_score
Xt = forward_selector.transform( X )
cross_val_score(ppl , Xt, y, scoring = 'accuracy', cv=5)

## exercise:

Run the SFS in backward direction. 

 * How many features do you get?
 * Use cross-validation to evaluate the selected set of variables. What cross-validated accuracy do you get?

--- **correction** ---

In [ ]:
%load solutions/solution_SFS.py

# Recursive Feature Elimination

https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFE.html#sklearn.feature_selection.RFE


This needs:
 * `estimator` : an model, **which needs to expose a coefficient or importance score**
 * `n_features_to_select` : number of features to select
 * `step`: how many, or which fraction of samples to eliminate on each step
 
You will also sometimes need to add:
 * `importance_getter` : function to return the importance OR string indicating the attribute to use to get the feature importance  (useful when you have a pipeline)
 
 
 


### Exercise 

Run recursive feature elimination the on TGCA BRCA data (were we have selected the top variant genes).
 * use a random forest classifier
 * eliminate 10% of features at each steps
 * select 10 features

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# ... your code here

--- **correction** ---

In [ ]:
# %load solutions/solution_RFE.py

---

### RFE with a pipeline

Just to showcase the syntax when you have a pipeline:

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

ppl = Pipeline([('scale' , StandardScaler()),
                ('model' , LogisticRegression())
               ])

rfe = RFE( ppl,
           n_features_to_select = 10,
           step = 0.1,
           importance_getter = 'named_steps.model.coef_')
%time rfe.fit(X,y)

In [ ]:
rfe.get_feature_names_out()

### RFECV

Having to specify the numberof features is not ideal.

[RFE-CV](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.RFECV.html#sklearn.feature_selection.RFECV) uses cross-validation to find a good number of features.



In [ ]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression

ppl = Pipeline([('scale' , StandardScaler()),
                ('model' , LogisticRegression())
               ])

rfe = RFECV( ppl,
             min_features_to_select= 1,
             step = 0.05,
             cv = 10,
             scoring = 'accuracy',
             importance_getter = 'named_steps.model.coef_')
%time rfe.fit(X,y)

In [ ]:
import matplotlib.pyplot as plt 

data = {
    key: rfe.cv_results_[key]
    for key in ["n_features", "mean_test_score", "std_test_score"]
}

cv_results = pd.DataFrame(data)
plt.figure()
plt.xlabel("Number of features selected")
plt.ylabel("Mean test accuracy")
plt.errorbar(
    x=cv_results["n_features"],
    y=cv_results["mean_test_score"],
    yerr=cv_results["std_test_score"],
)
plt.title("Recursive Feature Elimination")
plt.show()